In [42]:
# FIFA World Cup 2026 Prediction Engine
## Notebook 4: Monte Carlo Tournament Simulation
#This notebook simulates the entire FIFA World Cup 2026 tournament using a machine learning model trained on historical World Cup matches.
#A Monte Carlo framework is used to estimate championship probabilities, advancement probabilities, and likely final matchups.

In [43]:
## Load Model and Tournament Data

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import time
from openpyxl import load_workbook as load_wb

#Paths
WORKBOOK_PATH  = "WCup_2026_4.2.8_en.xlsx"  
PROCESSED_DIR  = "data/processed"
MODELS_DIR     = "models"
PREDICTIONS_DIR = "data/predictions"
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

#Random seed for reproducibility
MASTER_SEED = 2026
np.random.seed(MASTER_SEED)

print("Imports OK")
print(f"Workbook: {WORKBOOK_PATH}")

Imports OK
Workbook: WCup_2026_4.2.8_en.xlsx


In [2]:
wb = load_wb(WORKBOOK_PATH, read_only=True, data_only=True)

print("Sheets found:")
for s in wb.sheetnames:
    print(f"  {s}")

REQUIRED_SHEETS = ["Groups", "Matches", "AssignThird", "ThirdPlaced"]
missing = [s for s in REQUIRED_SHEETS if s not in wb.sheetnames]
if missing:
    raise ValueError(f"Required sheets missing from workbook: {missing}")

print(f"\nAll required sheets present: {REQUIRED_SHEETS}")

Sheets found:
  World Cup
  DailySchedule
  FairPlay
  HeadToHeadComp
  Predictions_1
  Predictions_Ranking_1
  Predictions_2
  Predictions_Ranking_2
  PrSettings
  Language
  TimeZone
  Help
  Groups
  Matches
  AssignThird
  ThirdPlaced
  Distinctness
  CalcA
  CalcB
  CalcC
  CalcD
  CalcE
  CalcF
  CalcG
  CalcH
  CalcI
  CalcJ
  CalcK
  CalcL

All required sheets present: ['Groups', 'Matches', 'AssignThird', 'ThirdPlaced']


In [3]:
best_model    = joblib.load(f"{MODELS_DIR}/best_model.pkl")
label_encoder = joblib.load(f"{MODELS_DIR}/label_encoder.pkl")

print(f"Model type:      {type(best_model).__name__}")
print(f"Label classes:   {label_encoder.classes_}")
print(f"Expected input:  {best_model.n_features_in_} features")

if hasattr(best_model, "feature_names_in_"):
    print("\nModel feature order:")
    for f in best_model.feature_names_in_:
        print(f"  {f}")

Model type:      RandomForestClassifier
Label classes:   ['Away Win' 'Draw' 'Home Win']
Expected input:  23 features

Model feature order:
  draws_last_4y_difference
  fifa_points_pre_tournament_difference
  fifa_rank_pre_tournament_difference
  finals_before_difference
  goals_received_last_4y_difference
  goals_scored_last_4y_difference
  groups_passed_before_difference
  is_host_difference
  losses_last_4y_difference
  quarterfinals_before_difference
  round16_before_difference
  semifinals_before_difference
  squad_avg_age_difference
  squad_total_market_value_eur_difference
  wins_last_4y_difference
  world_cup_participations_before_difference
  world_cup_titles_before_difference
  fifa_points_pre_tournament_ratio
  goals_scored_last_4y_ratio
  goals_received_last_4y_ratio
  wins_last_4y_ratio
  squad_avg_age_ratio
  host_difference


In [4]:
team_features = pd.read_csv(
    f"{PROCESSED_DIR}/team_features_2026_model_ready.csv"
).set_index("team")

print(f"team_features shape: {team_features.shape}")
print(f"Teams in index ({len(team_features)}):")
print(sorted(team_features.index.tolist()))

team_features shape: (48, 18)
Teams in index (48):
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia And Herzegovina', 'Brazil', 'Cabo Verde', 'Canada', 'Colombia', 'Congo DR', 'Croatia', 'Curaçao', 'Czechia', "Côte D'Ivoire", 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Türkiye', 'United States', 'Uruguay', 'Uzbekistan']


In [5]:
def parse_groups_sheet(wb):
    """
    Groups sheet layout (from inspection):
      col[1] = slot code  (e.g. 'A1', 'B3')
      col[2] = FIFA rank number
      col[3] = team name
      col[4] = 'Played' flag (True/False or None)
      A group separator row has col[3] = single letter ('A'..'L')

    Returns:
      slot_to_team : dict  e.g. {'A1': 'Mexico', 'A2': 'South Africa', ...}
      group_to_slots: dict e.g. {'A': ['A1','A2','A3','A4'], ...}
    """
    ws = wb["Groups"]
    slot_to_team   = {}
    group_to_slots = {}
    current_group  = None

    for row in ws.iter_rows(values_only=True):
        slot = row[1]
        name = row[3]

        if slot is None and name is None:
            continue

        
        if isinstance(name, str) and len(name) == 1 and name.isalpha() and name.isupper():
            current_group = name
            group_to_slots[current_group] = []
            continue

        
        if isinstance(slot, str) and len(slot) == 2 and slot[0].isalpha() and slot[1].isdigit():
            if isinstance(name, str) and current_group:
                slot_to_team[slot] = name
                group_to_slots[current_group].append(slot)

    return slot_to_team, group_to_slots


slot_to_team, group_to_slots = parse_groups_sheet(wb)

print("Group structure:")
for grp in sorted(group_to_slots.keys()):
    slots = group_to_slots[grp]
    teams = [slot_to_team[s] for s in slots]
    print(f"  Group {grp}: {dict(zip(slots, teams))}")

print(f"\nTotal slots: {len(slot_to_team)} (expected 48)")

Group structure:
  Group A: {'A1': 'Mexico', 'A2': 'South Africa', 'A3': 'Rep. of Korea', 'A4': 'Czech Rep.'}
  Group B: {'B1': 'Canada', 'B2': 'Bosnia/Herzeg.', 'B3': 'Qatar', 'B4': 'Switzerland'}
  Group C: {'C1': 'Brazil', 'C2': 'Morocco', 'C3': 'Haiti', 'C4': 'Scotland'}
  Group D: {'D1': 'USA', 'D2': 'Paraguay', 'D3': 'Australia', 'D4': 'Turkey'}
  Group E: {'E1': 'Germany', 'E2': 'Curaçao', 'E3': 'Ivory Coast', 'E4': 'Ecuador'}
  Group F: {'F1': 'Netherlands', 'F2': 'Japan', 'F3': 'Sweden', 'F4': 'Tunisia'}
  Group G: {'G1': 'Belgium', 'G2': 'Egypt', 'G3': 'IR Iran', 'G4': 'New Zealand'}
  Group H: {'H1': 'Spain', 'H2': 'Cape Verde', 'H3': 'Saudi Arabia', 'H4': 'Uruguay'}
  Group I: {'I1': 'France', 'I2': 'Senegal', 'I3': 'Iraq', 'I4': 'Norway'}
  Group J: {'J1': 'Argentina', 'J2': 'Algeria', 'J3': 'Austria', 'J4': 'Jordan'}
  Group K: {'K1': 'Portugal', 'K2': 'DR Congo', 'K3': 'Uzbekistan', 'K4': 'Colombia'}
  Group L: {'L1': 'England', 'L2': 'Croatia', 'L3': 'Ghana', 'L4': 'Pan

In [44]:
## Team Name Standardization

In [6]:
#matching names

WORKBOOK_TO_FEATURES = {
    "Rep. of Korea": "South Korea",
    "Czech Rep.": "Czechia",
    "Bosnia/Herzeg.": "Bosnia And Herzegovina",
    "Ivory Coast": "Côte D'Ivoire",
    "Curaçao": "Curaçao",
    "DR Congo": "Congo DR",
    "IR Iran": "Iran",
    "Cape Verde": "Cabo Verde",
    "Turkey": "Türkiye",
    "USA": "United States",
}

def normalize(workbook_name):
    """Translate a workbook team name to its team_features canonical name."""
    return WORKBOOK_TO_FEATURES.get(workbook_name, workbook_name)


# ── Validate every workbook team ──────────────────────────────
workbook_teams = sorted(set(slot_to_team.values()))
feature_teams  = set(team_features.index)

print(f"Workbook unique team names ({len(workbook_teams)}):")
unmapped = []
for wt in workbook_teams:
    canonical = normalize(wt)
    found = canonical in feature_teams
    status = "OK" if found else "MISSING"
    print(f"  {wt:30s} -> {canonical:30s}  [{status}]")
    if not found:
        unmapped.append((wt, canonical))

print(f"\n{'='*60}")
if unmapped:
    print(f"ACTION REQUIRED — {len(unmapped)} team(s) not mapped:")
    for wb_name, canon in unmapped:
        print(f"  '{wb_name}' -> tried '{canon}' — not in team_features")
    print("\nAdd entries to WORKBOOK_TO_FEATURES and re-run this cell.")
else:
    print("ALL TEAMS MAPPED SUCCESSFULLY")

Workbook unique team names (48):
  Algeria                        -> Algeria                         [OK]
  Argentina                      -> Argentina                       [OK]
  Australia                      -> Australia                       [OK]
  Austria                        -> Austria                         [OK]
  Belgium                        -> Belgium                         [OK]
  Bosnia/Herzeg.                 -> Bosnia And Herzegovina          [OK]
  Brazil                         -> Brazil                          [OK]
  Canada                         -> Canada                          [OK]
  Cape Verde                     -> Cabo Verde                      [OK]
  Colombia                       -> Colombia                        [OK]
  Croatia                        -> Croatia                         [OK]
  Curaçao                        -> Curaçao                         [OK]
  Czech Rep.                     -> Czechia                         [OK]
  DR Congo        

In [7]:
def parse_matches_sheet(wb):
    """
    Matches sheet columns (0-indexed from openpyxl row tuple):
      [0]  = None (padding)
      [1]  = Match number (int) OR stage label string (e.g. 'Round of 32')
      [2]  = Team 1 slot/ref  (e.g. 'A1', '1E', 'W74', 'RU101')
      [3]  = Team 2 slot/ref
      [4]  = Date (datetime)
      [7]  = Venue name
      [8]  = Team 1 name  (populated by workbook for known matches)
      [9]  = Team 2 name  (may be None for future knockout matches)
      [10] = R32 order number (for the R32 bracket sequence)

    Returns: list of match dicts
    """
    ws = wb["Matches"]
    matches = []
    current_stage = "Group"

    for row in ws.iter_rows(values_only=True):
        match_no = row[1]

        # Stage names 
        if isinstance(match_no, str):
            stage_raw = match_no.strip()
            if "32"    in stage_raw: current_stage = "R32"
            elif "16"  in stage_raw: current_stage = "R16"
            elif "1/4" in stage_raw: current_stage = "QF"
            elif "1/2" in stage_raw: current_stage = "SF"
            elif "Third" in stage_raw: current_stage = "3rd"
            elif "Final" in stage_raw: current_stage = "Final"
            continue

        if not isinstance(match_no, int):
            continue

        slot1 = row[2]
        slot2 = row[3]
        date  = row[4]
        venue = row[7]
        team1_name = row[8]   
        team2_name = row[9]   
        r32_order  = row[10]  

        matches.append({
            "match_no"   : match_no,
            "stage"      : current_stage,
            "slot1"      : str(slot1) if slot1 else None,
            "slot2"      : str(slot2) if slot2 else None,
            "date"       : date,
            "venue"      : venue,
            "wb_team1"   : team1_name,
            "wb_team2"   : team2_name,
            "r32_order"  : r32_order,
        })

    return matches


all_matches = parse_matches_sheet(wb)

# Summarize by stage
from collections import Counter
stage_counts = Counter(m["stage"] for m in all_matches)
print("Matches by stage:")
for stage, count in stage_counts.items():
    print(f"  {stage:10s}: {count}")

print(f"\nTotal matches: {len(all_matches)} (expected 104)")

# Show group matches
group_matches = [m for m in all_matches if m["stage"] == "Group"]
print(f"\nGroup stage matches: {len(group_matches)} (expected 72)")
print("Sample (first 5):")
for m in group_matches[:5]:
    print(f"  #{m['match_no']:3d}  {m['slot1']} vs {m['slot2']}  "
          f"[{m['wb_team1']} vs {m['wb_team2']}]  @ {m['venue']}")

Matches by stage:
  Group     : 72
  R32       : 16
  R16       : 8
  QF        : 4
  SF        : 2
  3rd       : 1
  Final     : 1

Total matches: 104 (expected 104)

Group stage matches: 72 (expected 72)
Sample (first 5):
  #  1  A1 vs A2  [Mexico vs South Africa]  @ Mexico City
  #  2  A3 vs A4  [Rep. of Korea vs Czech Rep.]  @ Guadalajara
  #  3  B1 vs B2  [Canada vs Bosnia/Herzeg.]  @ Toronto
  #  4  D1 vs D2  [USA vs Paraguay]  @ Los Angeles
  #  5  C3 vs C4  [Haiti vs Scotland]  @ Boston


In [8]:
# For each group, collecting 6 (slot1, slot2) pairs from group matches

group_fixtures = {g: [] for g in sorted(group_to_slots.keys())}

for m in group_matches:
    # Determine which group this match belongs to
    grp = m["slot1"][0]
    group_fixtures[grp].append((m["slot1"], m["slot2"], m["match_no"]))

print("Group fixtures (slot pairs):")
for grp in sorted(group_fixtures.keys()):
    print(f"  Group {grp} ({len(group_fixtures[grp])} fixtures):")
    for s1, s2, mno in group_fixtures[grp]:
        t1 = slot_to_team.get(s1, "?")
        t2 = slot_to_team.get(s2, "?")
        print(f"    #{mno:3d}: {s1}({t1}) vs {s2}({t2})")

Group fixtures (slot pairs):
  Group A (6 fixtures):
    #  1: A1(Mexico) vs A2(South Africa)
    #  2: A3(Rep. of Korea) vs A4(Czech Rep.)
    # 25: A4(Czech Rep.) vs A2(South Africa)
    # 28: A1(Mexico) vs A3(Rep. of Korea)
    # 53: A4(Czech Rep.) vs A1(Mexico)
    # 54: A2(South Africa) vs A3(Rep. of Korea)
  Group B (6 fixtures):
    #  3: B1(Canada) vs B2(Bosnia/Herzeg.)
    #  8: B3(Qatar) vs B4(Switzerland)
    # 26: B4(Switzerland) vs B2(Bosnia/Herzeg.)
    # 27: B1(Canada) vs B3(Qatar)
    # 51: B4(Switzerland) vs B1(Canada)
    # 52: B2(Bosnia/Herzeg.) vs B3(Qatar)
  Group C (6 fixtures):
    #  5: C3(Haiti) vs C4(Scotland)
    #  7: C1(Brazil) vs C2(Morocco)
    # 29: C1(Brazil) vs C3(Haiti)
    # 30: C4(Scotland) vs C2(Morocco)
    # 49: C4(Scotland) vs C1(Brazil)
    # 50: C2(Morocco) vs C3(Haiti)
  Group D (6 fixtures):
    #  4: D1(USA) vs D2(Paraguay)
    #  6: D3(Australia) vs D4(Turkey)
    # 31: D4(Turkey) vs D2(Paraguay)
    # 32: D1(USA) vs D3(Australia)
    # 59

In [46]:
#Assigning third position

In [9]:
def parse_assign_third(wb):
    """
    AssignThird encodes FIFA's official rule for which third-placed team
    goes to which R32 match slot, based on the combination of 8 groups
    that produced qualifying third-placed teams.

    Sheet layout (from inspection):
      Row 2 (header): cols 3..10 = R32 slot labels e.g. '3-CEFHI', '3-EFGIJ', ...
      Row 3: 'First in Group' for each slot (ignored)
      Row 4: Group letters that correspond to each slot (e.g. A, B, D, E...)
      Row 5: 'against third of Group' (ignored)
      Data rows: col[2] = group combination string (e.g. 'EFGHIJKL'),
                 cols[3..10] = which third group goes to each slot

    Returns:
      r32_slot_order  : list of 8 R32 slot labels (column headers)
      assign_table    : dict { group_combination_str -> { r32_slot -> group_letter } }
    """
    ws = wb["AssignThird"]
    rows = [row for row in ws.iter_rows(values_only=True)
            if any(v is not None for v in row)]

    header_row = rows[2]
    r32_slot_order = [str(v) for v in header_row[3:11] if v is not None]

    assign_table = {}
    for row in rows[6:]:   # data starts after header rows
        combo = row[2]
        if not isinstance(combo, str) or len(combo) != 8:
            continue
        #which group's third goes to each R32 slot
        assignments = {}
        for i, slot_label in enumerate(r32_slot_order):
            grp_letter = row[3 + i]
            if isinstance(grp_letter, str):
                assignments[slot_label] = grp_letter
        assign_table[combo] = assignments

    return r32_slot_order, assign_table


r32_slot_order, assign_table = parse_assign_third(wb)

print(f"R32 third-place slot labels ({len(r32_slot_order)}):")
print(f"  {r32_slot_order}")
print(f"\nAssignThird rows loaded: {len(assign_table)}")

# Show a sample
sample_combo = list(assign_table.keys())[0]
print(f"\nSample: combo='{sample_combo}'")
print(f"  {assign_table[sample_combo]}")

R32 third-place slot labels (8):
  ['3-CEFHI', '3-EFGIJ', '3-BEFIJ', '3-ABCDF', '3-AEHIJ', '3-CDFGH', '3-DEIJL', '3-EHIJK']

AssignThird rows loaded: 495

Sample: combo='EFGHIJKL'
  {'3-CEFHI': 'E', '3-EFGIJ': 'J', '3-BEFIJ': 'I', '3-ABCDF': 'F', '3-AEHIJ': 'H', '3-CDFGH': 'G', '3-DEIJL': 'L', '3-EHIJK': 'K'}


/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [45]:
## Match Feature Generation

In [10]:
MODEL_FEATURES = list(best_model.feature_names_in_) \
    if hasattr(best_model, "feature_names_in_") \
    else [
    "draws_last_4y_difference", "fifa_points_pre_tournament_difference",
    "fifa_rank_pre_tournament_difference", "finals_before_difference",
    "goals_received_last_4y_difference", "goals_scored_last_4y_difference",
    "groups_passed_before_difference", "is_host_difference",
    "losses_last_4y_difference", "quarterfinals_before_difference",
    "round16_before_difference", "semifinals_before_difference",
    "squad_avg_age_difference", "squad_total_market_value_eur_difference",
    "wins_last_4y_difference", "world_cup_participations_before_difference",
    "world_cup_titles_before_difference", "fifa_points_pre_tournament_ratio",
    "goals_scored_last_4y_ratio", "goals_received_last_4y_ratio",
    "wins_last_4y_ratio", "squad_avg_age_ratio", "host_difference",
]

print(f"Model expects {len(MODEL_FEATURES)} features:")
for f in MODEL_FEATURES:
    print(f"  {f}")

Model expects 23 features:
  draws_last_4y_difference
  fifa_points_pre_tournament_difference
  fifa_rank_pre_tournament_difference
  finals_before_difference
  goals_received_last_4y_difference
  goals_scored_last_4y_difference
  groups_passed_before_difference
  is_host_difference
  losses_last_4y_difference
  quarterfinals_before_difference
  round16_before_difference
  semifinals_before_difference
  squad_avg_age_difference
  squad_total_market_value_eur_difference
  wins_last_4y_difference
  world_cup_participations_before_difference
  world_cup_titles_before_difference
  fifa_points_pre_tournament_ratio
  goals_scored_last_4y_ratio
  goals_received_last_4y_ratio
  wins_last_4y_ratio
  squad_avg_age_ratio
  host_difference


In [11]:
def safe_ratio(a, b, cap=10.0):
    """
    Division that never produces inf or NaN.
    0/0 -> 1.0 (parity), x/0 -> ±cap, else a/b clipped to ±cap.
    """
    if b == 0 and a == 0:
        return 1.0
    if b == 0:
        return cap if a > 0 else -cap
    return float(np.clip(a / b, -cap, cap))


def create_match_features(team_a, team_b):
    """
    Build the exact 23-feature vector for team_a (home/team1)
    vs team_b (away/team2), in MODEL_FEATURES order.

    team_a, team_b : canonical names (keys in team_features index)
    Returns        : pd.DataFrame with 1 row and MODEL_FEATURES columns
    """
    row_a = team_features.loc[team_a]
    row_b = team_features.loc[team_b]
    values = []

    for feat in MODEL_FEATURES:
        if feat.endswith("_difference"):
            base = feat[:-len("_difference")]
            values.append(float(row_a[base]) - float(row_b[base]))
        elif feat.endswith("_ratio"):
            base = feat[:-len("_ratio")]
            values.append(safe_ratio(float(row_a[base]), float(row_b[base])))
        else:
            raise ValueError(f"Unknown feature suffix: {feat}")

    return pd.DataFrame([values], columns=MODEL_FEATURES)

In [12]:
# Pick two confirmed mapped teams
test_a = normalize("Germany")     
test_b = normalize("Brazil")      

feat_vec = create_match_features(test_a, test_b)

print(f"Feature vector for {test_a} vs {test_b}:")
print(feat_vec.T.to_string())

print(f"\nShape:   {feat_vec.shape}  (expected (1, {len(MODEL_FEATURES)}))")
print(f"Any NaN: {feat_vec.isnull().any().any()}")
print(f"Any inf: {np.isinf(feat_vec.values).any()}")
print(f"Column order matches model: {list(feat_vec.columns) == MODEL_FEATURES}")

Feature vector for Germany vs Brazil:
                                                    0
draws_last_4y_difference                     5.000000
fifa_points_pre_tournament_difference      -30.084313
fifa_rank_pre_tournament_difference          4.000000
finals_before_difference                     2.000000
goals_received_last_4y_difference           38.000000
goals_scored_last_4y_difference              9.000000
groups_passed_before_difference             -1.000000
is_host_difference                           0.000000
losses_last_4y_difference                    5.000000
quarterfinals_before_difference              0.000000
round16_before_difference                   -1.000000
semifinals_before_difference                 3.000000
squad_avg_age_difference                    -1.310000
squad_total_market_value_eur_difference      0.000000
wins_last_4y_difference                    -11.000000
world_cup_participations_before_difference  -2.000000
world_cup_titles_before_difference          

In [47]:
## Match Outcome Prediction

In [13]:
_prob_cache = {}   # (team_a, team_b) = {'Home Win': p, 'Draw': p, 'Away Win': p}


def predict_match_probabilities(team_a, team_b):
    """
    Returns win/draw/loss probabilities for team_a (home) vs team_b (away).
    Results are cached — identical matchups computed only once per session.
    """
    key = (team_a, team_b)
    if key in _prob_cache:
        return _prob_cache[key]

    X = create_match_features(team_a, team_b)
    proba = best_model.predict_proba(X)[0]

    result = {}
    for class_idx, encoded in enumerate(best_model.classes_):
        label = label_encoder.inverse_transform([encoded])[0]
        result[label] = float(proba[class_idx])

    _prob_cache[key] = result
    return result


def sample_outcome(probs):
    """
    Randomly sample one of 'Home Win' / 'Draw' / 'Away Win'
    according to model probabilities.
    This is the core Monte Carlo step.
    """
    labels = list(probs.keys())
    p = np.array(list(probs.values()))
    p = p / p.sum()   # guard against floating-point drift
    return np.random.choice(labels, p=p)

In [14]:
# Test 3 matchups with known-good team names
test_matchups = [
    (normalize("USA"),       normalize("Mexico")),
    (normalize("Germany"),   normalize("Brazil")),
    (normalize("Argentina"), normalize("France")),
]

for a, b in test_matchups:
    probs = predict_match_probabilities(a, b)
    total = sum(probs.values())
    print(f"{a:20s} vs {b:20s}")
    print(f"  Home Win={probs['Home Win']:.3f}  "
          f"Draw={probs['Draw']:.3f}  "
          f"Away Win={probs['Away Win']:.3f}  sum={total:.6f}")

# Verify sampling reproduces probabilities
print("\nSampling validation (10,000 draws for first matchup):")
p_test = predict_match_probabilities(*test_matchups[0])
samples = [sample_outcome(p_test) for _ in range(10_000)]
observed = pd.Series(samples).value_counts(normalize=True)
print(observed.round(3))
print("Expected:", {k: round(v, 3) for k, v in p_test.items()})

United States        vs Mexico              
  Home Win=0.290  Draw=0.294  Away Win=0.416  sum=1.000000
Germany              vs Brazil              
  Home Win=0.224  Draw=0.387  Away Win=0.389  sum=1.000000
Argentina            vs France              
  Home Win=0.268  Draw=0.421  Away Win=0.311  sum=1.000000

Sampling validation (10,000 draws for first matchup):
Away Win    0.411
Draw        0.299
Home Win    0.290
Name: proportion, dtype: float64
Expected: {'Away Win': 0.416, 'Draw': 0.294, 'Home Win': 0.29}


In [15]:
def proxy_goals(outcome, p_home, p_away):
    """
    The model predicts result probabilities only — not goals.
    Group standing tiebreakers need GD and GF, so we use a simple
    deterministic proxy based on the sampled outcome and the model's
    confidence margin.

    Margin thresholds:
      |p_home - p_away| > 0.35 -> 'dominant' win (2-0)
      otherwise                -> 'close'    win (2-1)
      Draw                     -> 1-1

    This is ONLY used for tiebreaking — it never feeds back into
    the model or affects win/draw/loss probabilities.
    """
    if outcome == "Draw":
        return 1, 1
    dominant = abs(p_home - p_away) > 0.35
    if outcome == "Home Win":
        return (2, 0) if dominant else (2, 1)
    else:   # Away Win
        return (0, 2) if dominant else (1, 2)

In [48]:
## Group Stage Simulation

In [16]:
def simulate_group(group_letter, live_slot_to_team):
    """
    Simulate all 6 round-robin matches for one group.

    live_slot_to_team : dict mapping slot -> canonical team name
                        (updated during knockout to track slot winners)

    Returns a DataFrame of standings (4 rows), sorted by:
      1. Points  2. GD proxy  3. GF proxy  4. Random (fair-play substitute)
    """
    fixtures = group_fixtures[group_letter]
    teams_in_group = [live_slot_to_team[s] for s in group_to_slots[group_letter]]

    stats = {
        t: {"points": 0, "gf": 0, "ga": 0, "wins": 0, "draws": 0, "losses": 0}
        for t in teams_in_group
    }

    for slot1, slot2, _ in fixtures:
        t1 = live_slot_to_team[slot1]
        t2 = live_slot_to_team[slot2]

        probs   = predict_match_probabilities(t1, t2)
        outcome = sample_outcome(probs)
        gh, ga  = proxy_goals(outcome, probs["Home Win"], probs["Away Win"])

        stats[t1]["gf"] += gh;  stats[t1]["ga"] += ga
        stats[t2]["gf"] += ga;  stats[t2]["ga"] += gh

        if outcome == "Home Win":
            stats[t1]["points"] += 3;  stats[t1]["wins"]   += 1
            stats[t2]["losses"] += 1
        elif outcome == "Away Win":
            stats[t2]["points"] += 3;  stats[t2]["wins"]   += 1
            stats[t1]["losses"] += 1
        else:
            stats[t1]["points"] += 1;  stats[t1]["draws"]  += 1
            stats[t2]["points"] += 1;  stats[t2]["draws"]  += 1

    df = pd.DataFrame(stats).T.copy()
    df["gd"]     = df["gf"] - df["ga"]
    df["group"]  = group_letter
    df["team"]   = df.index
    df["_rand"]  = np.random.random(len(df))
    df = df.sort_values(
        by=["points", "gd", "gf", "_rand"],
        ascending=False
    ).reset_index(drop=True)
    df["position"] = range(1, 5)
    df = df.drop(columns=["_rand"])
    return df

In [17]:
print(slot_to_team["A3"])
print(slot_to_team["D1"])
print(slot_to_team["G3"])

Rep. of Korea
USA
IR Iran


In [18]:
canonical_slot_to_team = {
    slot: normalize(team)
    for slot, team in slot_to_team.items()
}

In [19]:
print(canonical_slot_to_team["A3"])
print(canonical_slot_to_team["D1"])
print(canonical_slot_to_team["G3"])

South Korea
United States
Iran


In [20]:
np.random.seed(42)

test_grp = "A"

test_standings = simulate_group(
    test_grp,
    canonical_slot_to_team
)

print(f"Test standings — Group {test_grp}:")
print(
    test_standings[
        ["position", "team", "points", "gf", "ga", "gd",
         "wins", "draws", "losses"]
    ].to_string(index=False)
)

# Sanity checks
assert len(test_standings) == 4, "Group should contain 4 teams"

total_points = test_standings["points"].sum()

# 6 matches in a 4-team group:
# all draws = 12 points distributed
# all decisive = 18 points distributed
assert 12 <= total_points <= 18, (
    f"Impossible total points distributed: {total_points}"
)

assert (
    test_standings["wins"].sum()
    + test_standings["draws"].sum()
    + test_standings["losses"].sum()
) == 12, "Each team should have played 3 matches"

print("\nGroup simulation sanity checks passed.")
print(f"Total points distributed: {total_points}")

Test standings — Group A:
 position         team  points  gf  ga  gd  wins  draws  losses
        1  South Korea       7   5   3   2     2      1       0
        2       Mexico       5   4   3   1     1      2       0
        3      Czechia       3   4   5  -1     1      0       2
        4 South Africa       1   3   5  -2     0      1       2

Group simulation sanity checks passed.
Total points distributed: 16


In [49]:
## Qualification Rules and Third-Place Ranking

In [21]:
def determine_qualifiers(all_group_standings):
    """
    From 12 completed group standings, determine 32 qualifiers:
      - Top 2 from each group (24 teams)
      - Best 8 of the 12 third-placed teams (8 teams)

    Third-place ranking uses FIFA's tiebreaker:
      Pts > GD > GF > random
    (Fair play is not modelled — noted as v1 limitation)

    Returns:
      group_winners  : {'A': 'Mexico', ...}
      group_runners  : {'A': 'Rep. of Korea', ...}
      best_8_thirds  : {'A': 'Czech Rep.', 'B': 'Qatar', ...}   <- group:team
      third_combo    : 'BCEFGIJK'   <- sorted 8-letter group string for AssignThird
    """
    group_winners  = {}
    group_runners  = {}
    all_thirds     = {}   # group_letter -> (team, pts, gd, gf)

    for grp, standings in all_group_standings.items():
        group_winners[grp] = standings.iloc[0]["team"]
        group_runners[grp] = standings.iloc[1]["team"]
        third_row = standings.iloc[2]
        all_thirds[grp] = {
            "team":   third_row["team"],
            "points": third_row["points"],
            "gd":     third_row["gd"],
            "gf":     third_row["gf"],
            "rand":   np.random.random(),
        }

    # Sort all 12 third-placed teams — best 8 qualify
    thirds_sorted = sorted(
        all_thirds.items(),
        key=lambda x: (x[1]["points"], x[1]["gd"], x[1]["gf"], x[1]["rand"]),
        reverse=True
    )

    best_8 = {grp: data["team"] for grp, data in thirds_sorted[:8]}
    # Combination string = sorted group letters of the 8 qualifying groups
    third_combo = "".join(sorted(best_8.keys()))

    return group_winners, group_runners, best_8, third_combo

In [50]:
## Knockout Bracket Resolution

In [22]:
def resolve_r32_thirds(best_8_thirds, third_combo):
    """
    Use AssignThird table to map each third-placed team to its
    specific R32 slot label (e.g. '3-CEFHI').

    best_8_thirds : {group_letter: canonical_team_name}
    third_combo   : 8-letter sorted string e.g. 'BCEFGIJK'

    Returns: {r32_slot_label: canonical_team_name}
    """
    if third_combo not in assign_table:
        # Fallback: pick closest combo (edge case for unusual qualification)
        print(f"WARNING: combo '{third_combo}' not in AssignThird table. "
              f"Using closest available.")
        # Attempt exact match first, then find by set similarity
        combos = list(assign_table.keys())
        combo_set = set(third_combo)
        scored = sorted(
            combos,
            key=lambda c: len(set(c) & combo_set),
            reverse=True
        )
        third_combo_used = scored[0]
        print(f"  Using '{third_combo_used}' instead.")
    else:
        third_combo_used = third_combo

    assignments = assign_table[third_combo_used]
    # assignments = {r32_slot_label: group_letter}
    # We need to translate group_letter -> canonical team name
    return {
        slot_label: best_8_thirds[grp_letter]
        for slot_label, grp_letter in assignments.items()
        if grp_letter in best_8_thirds
    }

In [23]:
# Build the ordered R32 bracket from parsed Matches sheet.
# Each R32 match has:
#   match_no, slot1, slot2, r32_order (1..16, determines bracket pairing)

r32_matches_raw = sorted(
    [m for m in all_matches if m["stage"] == "R32"],
    key=lambda m: m["r32_order"] if isinstance(m["r32_order"], int) else 99
)

print(f"R32 matches ({len(r32_matches_raw)} expected 16):")
for m in r32_matches_raw:
    print(f"  R32-{m['r32_order']:>2d}  #{m['match_no']}  "
          f"{m['slot1']} vs {m['slot2']}")

R32 matches (16 expected 16):
  R32- 1  #73  2A vs 2B
  R32- 2  #76  1C vs 2F
  R32- 3  #74  1E vs 3-ABCDF
  R32- 4  #75  1F vs 2C
  R32- 5  #78  2E vs 2I
  R32- 6  #77  1I vs 3-CDFGH
  R32- 7  #79  1A vs 3-CEFHI
  R32- 8  #80  1L vs 3-EHIJK
  R32- 9  #82  1G vs 3-AEHIJ
  R32-10  #81  1D vs 3-BEFIJ
  R32-11  #84  1H vs 2J
  R32-12  #83  2K vs 2L
  R32-13  #85  1B vs 3-EFGIJ
  R32-14  #88  2D vs 2G
  R32-15  #86  1J vs 2H
  R32-16  #87  1K vs 3-DEIJL


In [24]:
def resolve_slot(
    slot_ref,
    group_winners,
    group_runners,
    best_8_thirds,
    third_to_r32_slot,
    knockout_winners   
):
    """
    Translates a slot reference string into a canonical team name.

    Slot reference types:
      'A1'       -> group slot (fixed, from Groups sheet)
      '1A'       -> group winner of group A
      '2A'       -> group runner-up of group A
      '3-CEFHI'  -> best 8 third: the third assigned to this R32 slot
      'W73'      -> winner of knockout match 73
      'RU101'    -> runner-up (loser) of semi-final 101

    Returns canonical team name or raises if unresolvable.
    """
    s = str(slot_ref).strip()

    # Group slot e.g. 'A1', 'B3'
    if len(s) == 2 and s[0].isalpha() and s[1].isdigit():
        return normalize(slot_to_team[s])

    # Group winner e.g. '1A'
    if len(s) == 2 and s[0].isdigit() and s[0] == "1":
        return group_winners[s[1]]

    # Group runner-up e.g. '2A'
    if len(s) == 2 and s[0].isdigit() and s[0] == "2":
        return group_runners[s[1]]

    # Third-place slot e.g. '3-CEFHI'
    if s.startswith("3-"):
        if s not in third_to_r32_slot:
            raise KeyError(f"Third slot '{s}' not in third_to_r32_slot: "
                           f"{third_to_r32_slot}")
        return third_to_r32_slot[s]

    # Knockout winner e.g. 'W73'
    if s.startswith("W") and s[1:].isdigit():
        mno = int(s[1:])
        if mno not in knockout_winners:
            raise KeyError(f"Winner of match {mno} not yet determined")
        return knockout_winners[mno]

    # Knockout runner-up e.g. 'RU101'
    if s.startswith("RU") and s[2:].isdigit():
        mno = int(s[2:])
        key = f"loser_{mno}"
        if key not in knockout_winners:
            raise KeyError(f"Loser of match {mno} not yet determined")
        return knockout_winners[key]

    raise ValueError(f"Cannot resolve slot reference: '{s}'")

In [52]:
#Knockout match simulation

In [25]:
def simulate_knockout_match(team_a, team_b):
    """
    Simulate a single knockout match.
    Returns (winner, loser, went_to_pens).

    If the model samples a Draw (valid in group stage, but not in
    knockout), we resolve via probability-weighted extra-time/pens:
      - renormalise P(Home Win) and P(Away Win)
      - sample the winner from those two probabilities only
    This preserves the model's relative preference between the two
    teams while ensuring a knockout match is always decided.
    """
    probs   = predict_match_probabilities(team_a, team_b)
    outcome = sample_outcome(probs)

    if outcome == "Home Win":
        return team_a, team_b, False
    elif outcome == "Away Win":
        return team_b, team_a, False
    else:
        # Draw -> extra time / penalties
        p_home = probs["Home Win"]
        p_away = probs["Away Win"]
        total  = p_home + p_away
        p_adj  = 0.5 if total == 0 else p_home / total
        if np.random.random() < p_adj:
            return team_a, team_b, True
        else:
            return team_b, team_a, True

In [51]:
## Full Tournament Simulation

In [26]:
def simulate_one_tournament():
    """
    Simulate one complete FIFA World Cup 2026.
    """

    # ── 1. GROUP STAGE ──────────────────────────────────────────
    all_group_standings = {}

    for grp in sorted(group_to_slots.keys()):
        all_group_standings[grp] = simulate_group(
            grp,
            canonical_slot_to_team
        )

    # ── 2. DETERMINE QUALIFIERS ──────────────────────────────────
    group_winners, group_runners, best_8_thirds, third_combo = \
        determine_qualifiers(all_group_standings)

    third_to_r32_slot = resolve_r32_thirds(
        best_8_thirds,
        third_combo
    )

    # ── Bracket state ────────────────────────────────────────────
    knockout_winners = {}

    def rs(slot_ref):
        return resolve_slot(
            slot_ref,
            group_winners,
            group_runners,
            best_8_thirds,
            third_to_r32_slot,
            knockout_winners
        )

    # ── 3. ROUND OF 32 ───────────────────────────────────────────
    r32_results = []

    for m in r32_matches_raw:
        t1 = rs(m["slot1"])
        t2 = rs(m["slot2"])

        w, l, pens = simulate_knockout_match(t1, t2)

        knockout_winners[m["match_no"]] = w

        r32_results.append({
            "match_no": m["match_no"],
            "t1": t1,
            "t2": t2,
            "winner": w,
            "pens": pens
        })

    # ── 4. ROUND OF 16 ───────────────────────────────────────────
    r16_matches = sorted(
        [m for m in all_matches if m["stage"] == "R16"],
        key=lambda m: m["match_no"]
    )

    r16_results = []

    for m in r16_matches:
        t1 = rs(m["slot1"])
        t2 = rs(m["slot2"])

        w, l, pens = simulate_knockout_match(t1, t2)

        knockout_winners[m["match_no"]] = w

        r16_results.append({
            "match_no": m["match_no"],
            "t1": t1,
            "t2": t2,
            "winner": w,
            "pens": pens
        })

    # ── 5. QUARTERFINALS ─────────────────────────────────────────
    qf_matches = sorted(
        [m for m in all_matches if m["stage"] == "QF"],
        key=lambda m: m["match_no"]
    )

    qf_results = []

    for m in qf_matches:
        t1 = rs(m["slot1"])
        t2 = rs(m["slot2"])

        w, l, pens = simulate_knockout_match(t1, t2)

        knockout_winners[m["match_no"]] = w

        qf_results.append({
            "match_no": m["match_no"],
            "t1": t1,
            "t2": t2,
            "winner": w,
            "pens": pens
        })

    # ── 6. SEMIFINALS ────────────────────────────────────────────
    sf_matches = sorted(
        [m for m in all_matches if m["stage"] == "SF"],
        key=lambda m: m["match_no"]
    )

    sf_results = []

    for m in sf_matches:
        t1 = rs(m["slot1"])
        t2 = rs(m["slot2"])

        w, l, pens = simulate_knockout_match(t1, t2)

        knockout_winners[m["match_no"]] = w
        knockout_winners[f"loser_{m['match_no']}"] = l

        sf_results.append({
            "match_no": m["match_no"],
            "t1": t1,
            "t2": t2,
            "winner": w,
            "loser": l,
            "pens": pens
        })

    # ── 7. THIRD PLACE MATCH ─────────────────────────────────────
    tp_match = [m for m in all_matches if m["stage"] == "3rd"][0]

    tp_t1 = rs(tp_match["slot1"])
    tp_t2 = rs(tp_match["slot2"])

    tp_w, tp_l, tp_pens = simulate_knockout_match(
        tp_t1,
        tp_t2
    )

    knockout_winners[tp_match["match_no"]] = tp_w

    # ── 8. FINAL ─────────────────────────────────────────────────
    final_match = [m for m in all_matches if m["stage"] == "Final"][0]

    fin_t1 = rs(final_match["slot1"])
    fin_t2 = rs(final_match["slot2"])

    fin_w, fin_l, fin_pens = simulate_knockout_match(
        fin_t1,
        fin_t2
    )

    knockout_winners[final_match["match_no"]] = fin_w

    # ── Correct advancement tracking ─────────────────────────────

    # Teams reaching Quarterfinals
    qf_teams = [r["winner"] for r in r16_results]

    # Teams reaching Semifinals
    sf_teams = [r["winner"] for r in qf_results]

    # Teams reaching Final
    fin_teams = [fin_t1, fin_t2]

    return {
        "champion": fin_w,
        "finalist_runner": fin_l,
        "finalists": fin_teams,
        "semifinalists": sf_teams,
        "quarterfinalists": qf_teams,
        "third_combo": third_combo,
        "r32_results": r32_results,
        "r16_results": r16_results,
        "qf_results": qf_results,
        "sf_results": sf_results,
        "final": {
            "t1": fin_t1,
            "t2": fin_t2,
            "winner": fin_w,
            "pens": fin_pens
        }
    }

In [53]:
## Monte Carlo Simulation Engine

In [27]:
N_SIM = 1000   

all_teams = sorted(team_features.index.tolist())
counts = {t: {"QF": 0, "SF": 0, "Final": 0, "Win": 0} for t in all_teams}
final_pairs  = {}
example_result = None
errors = []

np.random.seed(MASTER_SEED)
t0 = time.time()

for sim_i in range(N_SIM):
    try:
        result = simulate_one_tournament()
    except Exception as e:
        errors.append((sim_i, str(e)))
        if len(errors) <= 3:
            print(f"ERROR in simulation {sim_i}: {e}")
        continue

    for t in result["quarterfinalists"]:
        if t in counts: counts[t]["QF"]    += 1
    for t in result["semifinalists"]:
        if t in counts: counts[t]["SF"]    += 1
    for t in result["finalists"]:
        if t in counts: counts[t]["Final"] += 1
    if result["champion"] in counts:
        counts[result["champion"]]["Win"]   += 1

    pair = tuple(sorted(result["finalists"]))
    final_pairs[pair] = final_pairs.get(pair, 0) + 1

    if sim_i == 0:
        example_result = result

elapsed = time.time() - t0
success_count = N_SIM - len(errors)

print(f"Simulations: {N_SIM}")
print(f"Completed:   {success_count}")
print(f"Errors:      {len(errors)}")
print(f"Time:        {elapsed:.1f}s  ({elapsed/max(success_count,1):.2f}s per sim)")
print(f"Cache size:  {len(_prob_cache)} matchup probabilities cached")

Simulations: 1000
Completed:   1000
Errors:      0
Time:        36.1s  (0.04s per sim)
Cache size:  2001 matchup probabilities cached


In [28]:
print(example_result["champion"])
print(example_result["final"])
print(example_result["third_combo"])

Morocco
{'t1': 'France', 't2': 'Morocco', 'winner': 'Morocco', 'pens': False}
ABCEHIJL


In [29]:
for i in range(5):
    result = simulate_one_tournament()
    print(
        i,
        result["champion"],
        result["finalists"]
    )

0 Belgium ['Germany', 'Belgium']
1 Argentina ['Spain', 'Argentina']
2 Germany ['Germany', 'Mexico']
3 Belgium ['Belgium', 'Germany']
4 England ['Germany', 'England']


In [54]:
#Validatiojn Checks

In [30]:
print("VALIDATION 1 — Probability sums")
sample_keys = list(_prob_cache.keys())[:20]
fails = []
for key in sample_keys:
    total = sum(_prob_cache[key].values())
    if abs(total - 1.0) > 1e-6:
        fails.append((key, total))

if fails:
    print(f"FAIL — {len(fails)} matchups do not sum to 1.0:")
    for k, t in fails:
        print(f"  {k}: {t}")
else:
    print(f"PASS — checked {len(sample_keys)} matchups, all sum to 1.0")

VALIDATION 1 — Probability sums
PASS — checked 20 matchups, all sum to 1.0


In [31]:
print("VALIDATION 2 — Stage count integrity")
n_ok = success_count

total_wins   = sum(counts[t]["Win"]   for t in all_teams)
total_finals = sum(counts[t]["Final"] for t in all_teams)
total_sf     = sum(counts[t]["SF"]    for t in all_teams)
total_qf     = sum(counts[t]["QF"]    for t in all_teams)

checks = [
    ("Win  counts == N_OK",    total_wins,   n_ok),
    ("Final counts == N_OK*2", total_finals, n_ok * 2),
    ("SF   counts == N_OK*4",  total_sf,     n_ok * 4),
    ("QF   counts == N_OK*8",  total_qf,     n_ok * 8),
]

all_pass = True
for label, actual, expected in checks:
    status = "PASS" if actual == expected else "FAIL"
    all_pass = all_pass and (actual == expected)
    print(f"  {status}  {label}  (got {actual}, expected {expected})")

print(f"\nOverall: {'ALL PASS' if all_pass else 'FAILURES DETECTED'}")

VALIDATION 2 — Stage count integrity
  PASS  Win  counts == N_OK  (got 1000, expected 1000)
  PASS  Final counts == N_OK*2  (got 2000, expected 2000)
  PASS  SF   counts == N_OK*4  (got 4000, expected 4000)
  PASS  QF   counts == N_OK*8  (got 8000, expected 8000)

Overall: ALL PASS


In [32]:
print("VALIDATION 3 — Example bracket integrity (simulation #1)")

if example_result is None:
    print("No successful simulation to inspect.")
else:
    r32_winners_set = {r["winner"] for r in example_result["r32_results"]}
    r16_teams_set   = ({r["t1"] for r in example_result["r16_results"]}
                     | {r["t2"] for r in example_result["r16_results"]})
    r16_winners_set = {r["winner"] for r in example_result["r16_results"]}
    qf_teams_set    = ({r["t1"] for r in example_result["qf_results"]}
                     | {r["t2"] for r in example_result["qf_results"]})
    qf_winners_set  = {r["winner"] for r in example_result["qf_results"]}
    sf_teams_set    = ({r["t1"] for r in example_result["sf_results"]}
                     | {r["t2"] for r in example_result["sf_results"]})

    checks = [
        ("R32: 16 unique teams played",      len(r32_winners_set) == 16),
        ("R16: R16 teams ⊆ R32 winners",     r16_teams_set.issubset(r32_winners_set)),
        ("R16: 16 teams played",             len(r16_teams_set) == 16),
        ("QF:  QF  teams ⊆ R16 winners",     qf_teams_set.issubset(r16_winners_set)),
        ("QF:  8 teams played",              len(qf_teams_set) == 8),
        ("SF:  SF  teams ⊆ QF winners",      sf_teams_set.issubset(qf_winners_set)),
        ("SF:  4 teams played",              len(sf_teams_set) == 4),
        ("Final: champion in finalists",     example_result["champion"]
                                             in example_result["finalists"]),
        ("Final: 2 finalists",              len(example_result["finalists"]) == 2),
    ]

    for label, ok in checks:
        print(f"  {'PASS' if ok else 'FAIL'}  {label}")

    print(f"\nExample champion: {example_result['champion']}")
    print(f"Example finalists: {example_result['finalists']}")
    print(f"Third combo: {example_result['third_combo']}")

VALIDATION 3 — Example bracket integrity (simulation #1)
  PASS  R32: 16 unique teams played
  PASS  R16: R16 teams ⊆ R32 winners
  PASS  R16: 16 teams played
  PASS  QF:  QF  teams ⊆ R16 winners
  PASS  QF:  8 teams played
  PASS  SF:  SF  teams ⊆ QF winners
  PASS  SF:  4 teams played
  PASS  Final: champion in finalists
  PASS  Final: 2 finalists

Example champion: Morocco
Example finalists: ['France', 'Morocco']
Third combo: ABCEHIJL


In [55]:
#example tournament simulation

In [33]:
if example_result:
    print("EXAMPLE TOURNAMENT BRACKET (simulation #1)\n")

    print("── R32 ─────────────────────────────────────────────────")
    for r in example_result["r32_results"]:
        pens = " (ET/P)" if r["pens"] else ""
        print(f"  #{r['match_no']:3d}: {r['t1']:22s} vs {r['t2']:22s}  →  {r['winner']}{pens}")

    print("\n── R16 ─────────────────────────────────────────────────")
    for r in example_result["r16_results"]:
        pens = " (ET/P)" if r["pens"] else ""
        print(f"  #{r['match_no']:3d}: {r['t1']:22s} vs {r['t2']:22s}  →  {r['winner']}{pens}")

    print("\n── QF ──────────────────────────────────────────────────")
    for r in example_result["qf_results"]:
        pens = " (ET/P)" if r["pens"] else ""
        print(f"  #{r['match_no']:3d}: {r['t1']:22s} vs {r['t2']:22s}  →  {r['winner']}{pens}")

    print("\n── SF ──────────────────────────────────────────────────")
    for r in example_result["sf_results"]:
        pens = " (ET/P)" if r["pens"] else ""
        print(f"  #{r['match_no']:3d}: {r['t1']:22s} vs {r['t2']:22s}  →  {r['winner']}{pens}")

    print(f"\n── FINAL ───────────────────────────────────────────────")
    f = example_result["final"]
    pens = " (ET/P)" if f["pens"] else ""
    print(f"  {f['t1']:22s} vs {f['t2']:22s}  →  {f['winner']}{pens}")
    print(f"\n  🏆 WORLD CHAMPION: {example_result['champion']}")

EXAMPLE TOURNAMENT BRACKET (simulation #1)

── R32 ─────────────────────────────────────────────────
  # 73: Czechia                vs Bosnia And Herzegovina  →  Czechia
  # 76: Morocco                vs Japan                   →  Morocco
  # 74: Côte D'Ivoire          vs Brazil                  →  Brazil (ET/P)
  # 75: Sweden                 vs Scotland                →  Sweden (ET/P)
  # 78: Germany                vs Iraq                    →  Iraq
  # 77: France                 vs Saudi Arabia            →  France (ET/P)
  # 79: South Africa           vs Curaçao                 →  South Africa (ET/P)
  # 80: Ghana                  vs Norway                  →  Norway
  # 82: Belgium                vs South Korea             →  Belgium
  # 81: United States          vs Switzerland             →  Switzerland
  # 84: Uruguay                vs Austria                 →  Uruguay
  # 83: Colombia               vs England                 →  Colombia
  # 85: Canada                 vs Algeri

In [56]:
## Final Tournament Forecast

In [34]:
N_SIM = 10_000   

counts = {t: {"QF": 0, "SF": 0, "Final": 0, "Win": 0} for t in all_teams}
final_pairs = {}
errors = []

np.random.seed(MASTER_SEED)
t0 = time.time()

for sim_i in range(N_SIM):
    try:
        result = simulate_one_tournament()
    except Exception as e:
        errors.append((sim_i, str(e)))
        continue

    for t in result["quarterfinalists"]:
        if t in counts: counts[t]["QF"]    += 1
    for t in result["semifinalists"]:
        if t in counts: counts[t]["SF"]    += 1
    for t in result["finalists"]:
        if t in counts: counts[t]["Final"] += 1
    if result["champion"] in counts:
        counts[result["champion"]]["Win"]   += 1

    pair = tuple(sorted(result["finalists"]))
    final_pairs[pair] = final_pairs.get(pair, 0) + 1

    if sim_i % 1000 == 0:
        pct = sim_i / N_SIM * 100
        elapsed = time.time() - t0
        print(f"  {sim_i:6d}/{N_SIM}  ({pct:.0f}%)  {elapsed:.0f}s  "
              f"cache={len(_prob_cache)}")

elapsed = time.time() - t0
n_ok = N_SIM - len(errors)
print(f"\nDone: {n_ok}/{N_SIM} successful in {elapsed:.1f}s  "
      f"({elapsed/max(n_ok,1):.2f}s/sim)  errors={len(errors)}")

       0/10000  (0%)  0s  cache=2002
    1000/10000  (10%)  9s  cache=2002
    2000/10000  (20%)  19s  cache=2110
    3000/10000  (30%)  29s  cache=2159
    4000/10000  (40%)  38s  cache=2191
    5000/10000  (50%)  47s  cache=2203
    6000/10000  (60%)  56s  cache=2212
    7000/10000  (70%)  64s  cache=2220
    8000/10000  (80%)  73s  cache=2222
    9000/10000  (90%)  82s  cache=2230

Done: 10000/10000 successful in 90.8s  (0.01s/sim)  errors=0


In [57]:
## Championship Probabilities

In [35]:
n_ok = N_SIM - len(errors)

prob_table = pd.DataFrame([
    {
        "Team"            : t,
        "World Cup Win %" : round(counts[t]["Win"]   / n_ok * 100, 2),
        "Final %"         : round(counts[t]["Final"] / n_ok * 100, 2),
        "Semifinal %"     : round(counts[t]["SF"]    / n_ok * 100, 2),
        "Quarterfinal %"  : round(counts[t]["QF"]    / n_ok * 100, 2),
    }
    for t in all_teams
]).sort_values("World Cup Win %", ascending=False).reset_index(drop=True)

print(f"Championship probabilities ({n_ok} simulations):\n")
print(prob_table.to_string(index=False))

Championship probabilities (10000 simulations):

                  Team  World Cup Win %  Final %  Semifinal %  Quarterfinal %
             Argentina            12.53    19.13        28.73           41.31
                Brazil            11.25    18.31        28.26           42.36
                France            10.21    16.64        26.31           39.04
                 Spain             9.28    15.80        26.70           39.38
               England             8.34    14.31        24.39           39.64
               Germany             7.06    12.83        21.54           34.88
           Netherlands             4.90     9.73        17.44           31.37
               Belgium             4.45     9.66        18.45           33.05
              Portugal             2.94     6.55        13.31           26.41
               Uruguay             2.93     6.10        12.59           23.55
                Mexico             2.22     5.23        11.63           23.75
         United

In [58]:
#TOP 10 FAVORITES

In [36]:
print("TOP 10 FAVORITES\n")
print(prob_table.head(10).to_string(index=False))

TOP 10 FAVORITES

       Team  World Cup Win %  Final %  Semifinal %  Quarterfinal %
  Argentina            12.53    19.13        28.73           41.31
     Brazil            11.25    18.31        28.26           42.36
     France            10.21    16.64        26.31           39.04
      Spain             9.28    15.80        26.70           39.38
    England             8.34    14.31        24.39           39.64
    Germany             7.06    12.83        21.54           34.88
Netherlands             4.90     9.73        17.44           31.37
    Belgium             4.45     9.66        18.45           33.05
   Portugal             2.94     6.55        13.31           26.41
    Uruguay             2.93     6.10        12.59           23.55


In [60]:
#MOST LIKELY FINAL MATCHUPS

In [37]:
print("MOST LIKELY FINAL MATCHUPS (top 10)\n")
sorted_pairs = sorted(final_pairs.items(), key=lambda x: x[1], reverse=True)
for pair, cnt in sorted_pairs[:10]:
    pct = cnt / n_ok * 100
    print(f"  {pair[0]:25s} vs {pair[1]:25s}  →  {cnt:5d} ({pct:.2f}%)")

MOST LIKELY FINAL MATCHUPS (top 10)

  Argentina                 vs Spain                      →    209 (2.09%)
  Brazil                    vs France                     →    206 (2.06%)
  Argentina                 vs France                     →    188 (1.88%)
  Brazil                    vs Spain                      →    169 (1.69%)
  Argentina                 vs Brazil                     →    168 (1.68%)
  Argentina                 vs Germany                    →    163 (1.63%)
  England                   vs Spain                      →    160 (1.60%)
  England                   vs France                     →    147 (1.47%)
  Brazil                    vs Netherlands                →    136 (1.36%)
  Brazil                    vs Germany                    →    132 (1.32%)


In [62]:
#MOST LIKELY CHAMPION

In [38]:
champ_row = prob_table.iloc[0]
print(f"MOST LIKELY CHAMPION")
print(f"  {champ_row['Team']}  —  {champ_row['World Cup Win %']:.2f}% win probability")
print(f"  Final probability: {champ_row['Final %']:.2f}%")
print(f"  Semifinal probability: {champ_row['Semifinal %']:.2f}%")

MOST LIKELY CHAMPION
  Argentina  —  12.53% win probability
  Final probability: 19.13%
  Semifinal probability: 28.73%


In [39]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

prob_path = f"{PREDICTIONS_DIR}/world_cup_2026_team_probabilities.csv"
prob_table.to_csv(prob_path, index=False)
print(f"Saved: {prob_path}")

raw_counts = pd.DataFrame([
    {
        "Team"        : t,
        "QF_count"    : counts[t]["QF"],
        "SF_count"    : counts[t]["SF"],
        "Final_count" : counts[t]["Final"],
        "Win_count"   : counts[t]["Win"],
        "Simulations" : n_ok,
    }
    for t in all_teams
]).sort_values("Win_count", ascending=False).reset_index(drop=True)

mc_path = f"{PREDICTIONS_DIR}/world_cup_2026_monte_carlo_results.csv"
raw_counts.to_csv(mc_path, index=False)
print(f"Saved: {mc_path}")

Saved: data/predictions/world_cup_2026_team_probabilities.csv
Saved: data/predictions/world_cup_2026_monte_carlo_results.csv


In [40]:
print("FINAL VALIDATION — Monotonic advancement")
violations = prob_table[
    (prob_table["World Cup Win %"] > prob_table["Final %"]) |
    (prob_table["Final %"]         > prob_table["Semifinal %"]) |
    (prob_table["Semifinal %"]     > prob_table["Quarterfinal %"])
]
if violations.empty:
    print("PASS — Win% ≤ Final% ≤ Semifinal% ≤ Quarterfinal% for all 48 teams.")
else:
    print(f"FAIL — {len(violations)} violation(s):")
    print(violations.to_string(index=False))

FINAL VALIDATION — Monotonic advancement
PASS — Win% ≤ Final% ≤ Semifinal% ≤ Quarterfinal% for all 48 teams.


In [63]:
## Simulation Summary

In [41]:
print("=" * 60)
print("NOTEBOOK 04 SUMMARY")
print("=" * 60)
print(f"\nWorkbook:        {WORKBOOK_PATH}")
print(f"Sheets used:     Groups, Matches, AssignThird, ThirdPlaced")
print(f"Model:           {type(best_model).__name__} "
      f"({best_model.n_features_in_} features)")
print(f"Simulations:     {N_SIM}  ({n_ok} successful, {len(errors)} errors)")
print(f"Prob cache:      {len(_prob_cache)} unique matchup pairs")
print(f"\nPredicted Champion: {prob_table.iloc[0]['Team']} "
      f"({prob_table.iloc[0]['World Cup Win %']:.2f}%)")
print(f"Most likely Final:  {sorted_pairs[0][0][0]} vs {sorted_pairs[0][0][1]} "
      f"({sorted_pairs[0][1]/n_ok*100:.2f}%)")
print(f"\nOutputs saved to {PREDICTIONS_DIR}/")
print("  - world_cup_2026_team_probabilities.csv")
print("  - world_cup_2026_monte_carlo_results.csv")
print("\nDocumented approximations:")
print("  1. Proxy goals (2-0 / 2-1 / 1-1) for group tiebreakers only")
print("  2. Draw in knockout -> renormalized ET/pens probability")
print("  3. squad_total_market_value_eur = 0 for all teams (no source)")
print("  4. *_last_4y stats from 2022 snapshot (stale for 2026)")
print("  5. Fair play not modelled in third-place tiebreaker")

NOTEBOOK 04 SUMMARY

Workbook:        WCup_2026_4.2.8_en.xlsx
Sheets used:     Groups, Matches, AssignThird, ThirdPlaced
Model:           RandomForestClassifier (23 features)
Simulations:     10000  (10000 successful, 0 errors)
Prob cache:      2233 unique matchup pairs

Predicted Champion: Argentina (12.53%)
Most likely Final:  Argentina vs Spain (2.09%)

Outputs saved to data/predictions/
  - world_cup_2026_team_probabilities.csv
  - world_cup_2026_monte_carlo_results.csv

Documented approximations:
  1. Proxy goals (2-0 / 2-1 / 1-1) for group tiebreakers only
  2. Draw in knockout -> renormalized ET/pens probability
  3. squad_total_market_value_eur = 0 for all teams (no source)
  4. *_last_4y stats from 2022 snapshot (stale for 2026)
  5. Fair play not modelled in third-place tiebreaker
